# Notebook 00: Obtención y Preprocesamiento del Dataset

Este notebook documenta el proceso de obtención de datos desde IEDB y el preprocesamiento
realizado para generar el archivo filtrado que usará el Notebook 01.

**Entrada:** `data/raw/tcell_full_v3.csv` y `data/raw/bcell_full_v3.csv` (descargados por `download_files.ipynb`)  
**Salida:** `data/processed/iedb_sars_flu_filtered.csv`

---

## Índice
1. Fuente de datos: IEDB
2. Primer intento: `epitope_full_v3.csv` y por qué no sirve
3. Decisión: usar `tcell_full_v3.csv` y `bcell_full_v3.csv`
4. Scripts de filtrado
5. Resultado del preprocesamiento
6. Decisiones documentadas para el Notebook 02

---
## 1. Fuente de datos: IEDB

**IEDB (Immune Epitope Database)** es la mayor base de datos pública de epítopos inmunológicos
del mundo. Está mantenida por el National Institute of Allergy and Infectious Diseases (NIAID)
de Estados Unidos y contiene datos experimentales acumulados durante décadas de investigación.

URL: [https://www.iedb.org](https://www.iedb.org)  
Exportaciones: [https://www.iedb.org/database_export_v3.php](https://www.iedb.org/database_export_v3.php)

### ¿Por qué IEDB?

- Es gratuita y descargable sin restricciones
- Contiene datos experimentales reales, no simulaciones
- Cubre múltiples patógenos incluyendo SARS-CoV-2 e Influenza A
- Cada registro incluye el resultado del ensayo (Positive / Negative)
- Es la fuente que usan los investigadores reales en inmunología computacional

### Concepto clave: epítopo

Un **epítopo** es el fragmento específico de una proteína que el sistema inmune aprende
a reconocer. Típicamente son péptidos de 8-15 aminoácidos. Si una proteína contiene
epítopos validados experimentalmente, hay evidencia real de que el sistema inmune humano
la detecta: esa proteína es, por definición, antigénica.

IEDB cataloga estos epítopos junto con los resultados de los ensayos de laboratorio
que los identificaron. Esa información es exactamente lo que necesitamos para construir
nuestras etiquetas de entrenamiento (`label = 1` antigénica, `label = 0` no antigénica).

---
## 2. Primer intento: `epitope_full_v3.csv` y por qué no sirve

El primer archivo que descargamos fue `epitope_full_v3.csv` (~830 MB descomprimido),
que parecía la opción natural al ser la exportación completa de epítopos.

### Problema descubierto

**1. Doble cabecera.** El CSV tiene dos filas de cabecera:
- Fila 0: agrupadores de sección genéricos (`Epitope`, `Epitope.1`, `Related Object`...)
- Fila 1: nombres descriptivos reales (`Source Organism`, `Source Molecule`...)

Esto requiere cargarlo con `header=1` en pandas, y aun así los nombres de columna
aparecen duplicados porque el mismo nombre se repite en distintas secciones del archivo.

**2. Sin columna de resultado de ensayo.** El archivo `epitope_full_v3.csv` describe
la estructura de cada epítopo (secuencia, proteína fuente, organismo...) pero **no incluye
el resultado del ensayo** (Positive / Negative). Ese dato crítico para nuestro proyecto
no está en este archivo.

### Conclusión

`epitope_full_v3.csv` no es útil para nuestro proyecto porque no contiene las etiquetas
de antigenicidad. Necesitamos los archivos de ensayos.

---
## 3. Decisión: usar `tcell_full_v3.csv` y `bcell_full_v3.csv`

IEDB organiza sus ensayos en dos grandes categorías:

| Archivo | Contenido | Tamaño descomprimido |
|---|---|---|
| `tcell_full_v3.csv` | Ensayos de respuesta de células T | ~1.3 GB |
| `bcell_full_v3.csv` | Ensayos de respuesta de células B (anticuerpos) | ~2.6 GB |

Ambos archivos incluyen:
- El epítopo ensayado (secuencia, proteína fuente, organismo)
- El resultado del ensayo: `Positive`, `Negative`, `Positive-High`, `Positive-Low`, `Positive-Intermediate`

### ¿Por qué usar ambos?

La inmunidad frente a un patógeno implica tanto la respuesta celular (células T) como
la respuesta humoral (células B / anticuerpos). Una proteína puede ser antigénica
para uno o ambos tipos de respuesta. Usar solo uno de los dos archivos dejaría fuera
evidencia experimental relevante.

### Problema de tamaño

Los archivos descomprimidos son demasiado grandes para cargarlos en memoria de golpe.
La solución es filtrar por chunks, extrayendo solo las columnas y filas que necesitamos.

---
## 4. Scripts de filtrado

### Columnas seleccionadas

De los ~160 columnas disponibles en cada archivo, solo necesitamos 5:

| Columna | Posición en tcell | Posición en bcell | Descripción |
|---|---|---|---|
| `epitope_name` | 11 | 11 | Secuencia o nombre del epítopo |
| `source_molecule` | 19 | 19 | Nombre de la proteína fuente |
| `source_molecule_iri` | 20 | 20 | Identificador UniProt de la proteína |
| `source_organism` | 23 | 23 | Nombre del organismo fuente |
| `qualitative_measurement` | 122 | 102 | Resultado del ensayo |

### Script 1: Filtrado de tcell

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

In [ ]:
ORGANISMS = ['SARS-CoV-2', 'Severe acute respiratory syndrome coronavirus 2', 'Influenza A']

COLS_TCELL = [11, 19, 20, 23, 122]
COL_NAMES  = ['epitope_name', 'source_molecule', 'source_molecule_iri',
               'source_organism', 'qualitative_measurement']

print("Procesando tcell_full_v3.csv...")
chunks = []
for chunk in pd.read_csv(
    RAW_DIR / 'tcell_full_v3.csv',
    low_memory=False,
    header=1,
    usecols=COLS_TCELL,
    chunksize=100_000
):
    chunk.columns = COL_NAMES
    mask = chunk['source_organism'].str.contains(
        '|'.join(ORGANISMS), case=False, na=False
    )
    chunks.append(chunk[mask])

df_tcell = pd.concat(chunks, ignore_index=True)
df_tcell['assay_type'] = 'tcell'
print(f"  Filas filtradas: {len(df_tcell):,}")

### Script 2: Filtrado de bcell

In [ ]:
COLS_BCELL = [11, 19, 20, 23, 102]

print("Procesando bcell_full_v3.csv...")
chunks = []
for chunk in pd.read_csv(
    RAW_DIR / 'bcell_full_v3.csv',
    low_memory=False,
    header=1,
    usecols=COLS_BCELL,
    chunksize=100_000
):
    chunk.columns = COL_NAMES
    mask = chunk['source_organism'].str.contains(
        '|'.join(ORGANISMS), case=False, na=False
    )
    chunks.append(chunk[mask])

df_bcell = pd.concat(chunks, ignore_index=True)
df_bcell['assay_type'] = 'bcell'
print(f"  Filas filtradas: {len(df_bcell):,}")

### Script 3: Combinación y guardado final

In [ ]:
df = pd.concat([df_tcell, df_bcell], ignore_index=True)

OUTPUT_PATH = PROCESSED_DIR / 'iedb_sars_flu_filtered.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f"Guardado: {OUTPUT_PATH}")
print(f"\nResumen:")
print(f"  tcell:  {len(df_tcell):>7,} filas")
print(f"  bcell:  {len(df_bcell):>7,} filas")
print(f"  total:  {len(df):>7,} filas")
print(f"  Tamaño: {OUTPUT_PATH.stat().st_size / 1_048_576:.1f} MB")

---
## 5. Resultado del preprocesamiento

El archivo `iedb_sars_flu_filtered.csv` contiene **~158,289 filas** y **6 columnas**:

| Columna | Descripción |
|---|---|
| `epitope_name` | Secuencia o nombre del epítopo |
| `source_molecule` | Nombre de la proteína fuente |
| `source_molecule_iri` | Identificador UniProt de la proteína |
| `source_organism` | Nombre del organismo fuente |
| `qualitative_measurement` | Resultado del ensayo |
| `assay_type` | Tipo de ensayo (`tcell` o `bcell`) |

### Distribución de resultados de ensayo

| Valor | Filas | Interpretación |
|---|---|---|
| `Negative` | 88,418 | No antigénico en este ensayo |
| `Positive` | 56,663 | Antigénico confirmado |
| `Positive-Low` | 9,386 | Antigénico con respuesta baja |
| `Positive-High` | 2,073 | Antigénico con respuesta alta |
| `Positive-Intermediate` | 1,749 | Antigénico con respuesta intermedia |

### Organismos incluidos

- **SARS-CoV-2**: múltiples cepas y variantes (Wuhan/Hu-1, USA/CA, Omicron, etc.)
- **Influenza A**: múltiples cepas (H1N1, H3N2, H5N1, etc.)

---
## 6. Decisiones documentadas para el Notebook 02

### Definición de label = 1 (antigénico)

Una proteína recibe `label = 1` si tiene **al menos un epítopo con resultado positivo**
en cualquiera de los dos tipos de ensayo (tcell o bcell).

Los valores `Positive`, `Positive-Low`, `Positive-High` y `Positive-Intermediate`
se tratan todos como positivos, ya que todos representan evidencia experimental
de reconocimiento por el sistema inmune.

### Definición de label = 0 (no antigénico)

Una proteína recibe `label = 0` si **solo tiene epítopos con resultado negativo**
y ninguno positivo en ninguno de los dos tipos de ensayo.

### Unidad de análisis: proteína, no epítopo

El dataset final tendrá **una fila por proteína**, no una fila por epítopo.
La agrupación se hará por `source_molecule_iri` (identificador de la proteína),
que es el identificador más estable y sin ambigüedades.

### Tratamiento de proteínas con ensayos mixtos

Es posible que una proteína tenga tanto epítopos positivos como negativos.
En ese caso, la proteína se etiqueta como `label = 1` porque hay evidencia
experimental de antigenicidad, aunque sea parcial.

### Archivo de salida

`data/processed/dataset.csv` — una fila por proteína, con sus features calculadas y su label.